In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local nocodb")
else:
    print("using aws nocodb")

using aws nocodb


In [3]:
from birddog.database import Database
from birddog.runtime import Runtime
from birddog.database_updater import DatabaseUpdater
from birddog.wiki import (
    get_root_label,
    page_label,
    sequential_page_label,
    )

2026-07-08 11:27:35,179 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-07-08 11:27:35,188 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-07-08 11:27:35,372 [INFO] Translation is enabled. Using GCP translator
2026-07-08 11:27:35,373 [INFO] Using Google Cloud translation API
2026-07-08 11:27:35,373 [INFO] GoogleCloudTranslator using REST API


In [4]:
db = Database()

2026-07-08 11:28:04,249 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-07-08 11:28:04,506 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     3.93    39.00       0.00           24


In [5]:
recs, _ = db.scan("Documents", view_name="Nonempty Hash", fields=["url", "sha1_hash", "timestamp"])

In [6]:
recs[0]

{'Id': 8295,
 'url': 'https://commons.wikimedia.org/wiki/File:ДАЗпО_Р-5593-24-53_Книга_реєстрації_актів_цивільного_стану_про_народження_(1936).pdf',
 'timestamp': '2026-04-16 00:00:00+00:00',
 'sha1_hash': 'ddfdeb295dbeda425c3b095102096a0dff561cd9'}

In [7]:
def get_all_hashed_docs(db):
    cursor = None
    limit = 1000
    result = []
    while True:
        recs, cursor = db.scan(
            "Documents", 
            view_name="Nonempty Hash", 
            fields=["url", "sha1_hash", "timestamp"],
            cursor=cursor,
            limit=limit)
        if not recs:
            break
        result.extend(recs)
        print(len(result))
        if not cursor:
            break
    return result

In [8]:
docs = get_all_hashed_docs(db)

2026-07-08 11:40:21,723 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   22.00     0.01    39.00       0.00           24
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
2026-07-08 11:41:21,919 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api

In [9]:
docs[0]

{'Id': 8295,
 'url': 'https://commons.wikimedia.org/wiki/File:ДАЗпО_Р-5593-24-53_Книга_реєстрації_актів_цивільного_стану_про_народження_(1936).pdf',
 'timestamp': '2026-04-16 00:00:00+00:00',
 'sha1_hash': 'ddfdeb295dbeda425c3b095102096a0dff561cd9'}

In [10]:
sha_dict = {}
for doc in docs:
    key = doc["sha1_hash"]
    items = sha_dict.get(key, [])
    items.append(doc)
    sha_dict[key] = items

In [14]:
counts = {}
for k, v in sha_dict.items():
    n = len(v)
    if n > 1:
        counts[n] = counts.get(n, 0) + 1
    if n > 5:
        print([r["url"] for r in v])

['https://uk.wikisource.org/wiki/Файл:ДАЖО_67-3-630._1903-1918._Метрична_книга_євреїв_м._Котельня._Смерть.pdf', 'https://commons.wikimedia.org/wiki/File:ДАЖО_67-3-630._1903-1918._Метрична_книга_євреїв_м._Котельня._Смерть.pdf', 'https://uk.wikisource.org/wiki/File:ДАЖО_67-3-630._1903-1918._Метрична_книга_євреїв_м._Котельня._Смерть.pdf', 'https://uk.wikisource.org/wiki/File:ДАЖО_67-3-630._1903-1918._Метричні_книги_єврейської_громади_містечка_Котельня_про_cмерть.pdf', 'https://commons.wikimedia.org/wiki/File:ДАЖО_67-3-630._1903-1918._Метричні_книги_єврейської_громади_містечка_Котельня_про_cмерть.pdf', 'https://commons.wikimedia.org/wiki/File:ДАЖО_67-3-630._1903-1918._Метричні_книги_єврейської_громади_містечка_Котельня_про_смерть.pdf', 'https://uk.wikisource.org/wiki/File:ДАЖО_67-3-630._1903-1918._Метричні_книги_єврейської_громади_містечка_Котельня_про_смерть.pdf']
['https://uk.wikisource.org/wiki/Файл:ДАЖО_67-3-629._1902-1918._Метрична_книга_євреїв_м._Котельня._Смерть.pdf', 'https://commo

In [13]:
counts

{2: 14803, 3: 1069, 4: 42, 5: 32, 7: 2, 6: 1}